In [1]:
# ======================================================================
# ResGCN Input Ablation Study (COMPLETE – FINAL, CAMERA-READY)
# Target            : Aluminium
# Features          : Aluminium, Copper, Nickel, Zinc, Tin, Lead
# Experiments       : Lookback × Horizon × Input_Set (ALL subsets)
# Outputs           :
#   - Train & Val loss (printed every 5 epochs + plotted)
#   - Predictions (saved)
#   - Metrics (MAE, RMSE, MAPE, sMAPE, NRMSE, DA, AUHE, timing, complexity)
#   - Step-wise MAPE (per horizon step, plotted + saved)
#   - Diebold–Mariano test vs FULL
# ======================================================================

import time
import numpy as np
import pandas as pd
from pathlib import Path
from itertools import combinations, product
from scipy.stats import norm
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    mean_absolute_percentage_error
)

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau

from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GCNConv, LayerNorm


# ===================== Hyperparameters =====================
HIDDEN_CHANNELS = 32
NUM_LAYERS      = 3
DROPOUT         = 0.15
EPOCHS          = 200
LR              = 5e-4
WEIGHT_DECAY    = 1e-4
PATIENCE        = 15
BATCH_SIZE      = 8

LOOKBACKS = [10, 22, 63]
HORIZONS  = [1, 5, 10, 22, 63]

TARGET = "Aluminium"
ALL_FEATURES = ['Aluminium', 'Copper', 'Nickel', 'Zinc', 'Tin', 'Lead']

# ===================== Reproducibility =====================
def set_seed(seed=42):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(123)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# ===================== Metrics =====================
def directional_accuracy(y_true, y_pred):
    if len(y_true) < 2:
        return np.nan
    return float((np.sign(np.diff(y_true)) == np.sign(np.diff(y_pred))).mean())

def nrmse(y_true, y_pred):
    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))
    return float(rmse / (np.max(y_true) - np.min(y_true) + 1e-12))

def smape(y_true, y_pred):
    denom = (np.abs(y_true) + np.abs(y_pred)) / 2.0
    return float(np.mean(np.abs(y_true - y_pred) / (denom + 1e-12)) * 100)


def estimate_gcn_flops(num_nodes, seq_len, hidden, horizon, num_layers):
    """
    Approximate FLOPs for ResGCNNode0
    Assumes fully-connected graph (worst case)
    """

    flops = 0

    # Temporal projection: (N nodes × seq_len × hidden)
    flops += num_nodes * seq_len * hidden

    # GCN layers
    for _ in range(num_layers):
        # Message passing: N^2 × hidden
        flops += num_nodes * num_nodes * hidden
        # Linear transform
        flops += num_nodes * hidden * hidden

    # Head
    flops += hidden * (hidden // 2)
    flops += (hidden // 2) * horizon

    return float(flops)



# ===================== AUHE =====================
def compute_auhe(step_mapes):
    """
    Area Under Horizon Error (AUHE)
    Trapezoidal integration over forecast steps
    """
    steps = np.arange(1, len(step_mapes) + 1)
    return float(np.trapz(step_mapes, steps))

# ===================== Diebold–Mariano =====================
def diebold_mariano(e1, e2):
    d = e1**2 - e2**2
    dm_stat = np.mean(d) / np.sqrt(np.var(d, ddof=1) / len(d))
    p_value = 2 * (1 - norm.cdf(abs(dm_stat)))
    return dm_stat, p_value

# ===================== Input Sets =====================
def generate_input_sets(all_features, target):
    others = [f for f in all_features if f != target]
    input_sets = {}
    for r in range(len(others) + 1):
        for combo in combinations(others, r):
            feats = [target] + list(combo)
            tag = "FULL" if len(feats) == len(all_features) \
                else "w/o_" + "_".join(sorted(set(others) - set(combo)))
            input_sets[tag] = feats
    return input_sets

INPUT_SETS = generate_input_sets(ALL_FEATURES, TARGET)

# ===================== Data =====================
df = pd.read_csv("Metals_Price.csv", parse_dates=["Date"])
df.sort_values("Date", inplace=True)

dates = df["Date"].values
values_all = df[ALL_FEATURES].values.astype(np.float32)

# ===================== Graph =====================
def build_edge_index(n):
    edges = [[i, j] for i in range(n) for j in range(n)]
    return torch.tensor(edges, dtype=torch.long).t().contiguous()

# ===================== Model =====================
class ResidualGCNBlock(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.conv = GCNConv(ch, ch)
        self.norm = LayerNorm(ch)

    def forward(self, x, edge_index):
        h = self.conv(x, edge_index)
        h = F.relu(self.norm(h))
        h = F.dropout(h, p=DROPOUT, training=self.training)
        return x + h

class ResGCNNode0(nn.Module):
    def __init__(self, seq_len, hidden, horizon):
        super().__init__()
        self.temporal = nn.Linear(seq_len, hidden)
        self.blocks = nn.ModuleList([ResidualGCNBlock(hidden) for _ in range(NUM_LAYERS)])
        self.head = nn.Sequential(
            nn.Linear(hidden, hidden // 2),
            nn.ReLU(),
            nn.Dropout(DROPOUT),
            nn.Linear(hidden // 2, horizon)
        )

    def forward(self, data):
        x = self.temporal(data.x.float())
        for blk in self.blocks:
            x = blk(x, data.edge_index)

        # batch shape: [num_nodes_total]
        # node-0 index for each graph:
        node0_mask = torch.zeros_like(data.batch, dtype=torch.bool)
        node0_mask[data.ptr[:-1]] = True  # ptr gives graph boundaries

        x0 = x[node0_mask]               # [batch_size, hidden]
        return self.head(x0)             # [batch_size, horizon]

# ===================== Windowing =====================
def create_windows(vals, dts, L, H, tgt):
    X, y, ts = [], [], []
    for i in range(L, len(vals) - H + 1):
        X.append(vals[i-L:i])
        y.append(vals[i:i+H, tgt])
        ts.append(dts[i:i+H])
    return np.array(X), np.array(y), np.array(ts)

def build_graphs(X, y, edge_index):
    return [Data(x=torch.tensor(xi.T), edge_index=edge_index, y=torch.tensor(yi)) for xi, yi in zip(X, y)]

# ===================== Model Stats =====================
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def model_size_mb(model):
    return count_parameters(model) * 4 / (1024 ** 2)

# ===================== Results =====================
ROOT = Path("results_resgcn_aluminium_ablation_full")
ROOT.mkdir(exist_ok=True)

STEP_PLOT_DIR = ROOT / "stepwise_mape_plots"
STEP_PLOT_DIR.mkdir(exist_ok=True)

all_results = []
dm_rows = []
reference_errors = {}

# ===================== Main Loop =====================
for L, H in product(LOOKBACKS, HORIZONS):
    print(f"\n=== Lookback {L} | Horizon {H} ===")

    for tag, FEATURES in INPUT_SETS.items():
        print(f"▶ Input: {tag}")

        feat_idx = [ALL_FEATURES.index(f) for f in FEATURES]
        tgt_idx = FEATURES.index(TARGET)

        EXP = ROOT / f"{tag}_LB{L}_H{H}"
        EXP.mkdir(parents=True, exist_ok=True)

        vals = values_all[:, feat_idx]
        total = len(vals) - L - H + 1
        tr_n, va_n = int(0.7 * total), int(0.15 * total)

        scaler = MinMaxScaler()
        scaler.fit(vals[:tr_n + L])
        vals_s = scaler.transform(vals)

        X, y, _ = create_windows(vals_s, dates, L, H, tgt_idx)
        Xtr, ytr = X[:tr_n], y[:tr_n]
        Xva, yva = X[tr_n:tr_n+va_n], y[tr_n:tr_n+va_n]
        Xte, yte = X[tr_n+va_n:], y[tr_n+va_n:]

        edge_index = build_edge_index(len(FEATURES)).to(device)

        train_loader = DataLoader(build_graphs(Xtr, ytr, edge_index), batch_size=BATCH_SIZE, shuffle=True)
        val_loader   = DataLoader(build_graphs(Xva, yva, edge_index), batch_size=BATCH_SIZE)
        test_loader  = DataLoader(build_graphs(Xte, yte, edge_index), batch_size=BATCH_SIZE)

        model = ResGCNNode0(L, HIDDEN_CHANNELS, H).to(device)
        opt = Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
        sch = ReduceLROnPlateau(opt, patience=5)
        crit = nn.MSELoss()

        train_losses, val_losses = [], []
        best, wait = np.inf, 0
        t0 = time.time()

        # ---------- Training ----------
        for ep in range(1, EPOCHS + 1):
            model.train()
            tl = []
            for b in train_loader:
                b = b.to(device)
                opt.zero_grad()
                loss = crit(model(b), b.y.view(-1, H))
                loss.backward()
                opt.step()
                tl.append(loss.item())

            model.eval()
            vl = []
            with torch.no_grad():
                for b in val_loader:
                    b = b.to(device)
                    vl.append(crit(model(b), b.y.view(-1, H)).item())

            train_losses.append(np.mean(tl))
            val_losses.append(np.mean(vl))
            sch.step(val_losses[-1])

            if ep % 5 == 0:
                print(f"Epoch {ep:03d} | Train {train_losses[-1]:.6f} | Val {val_losses[-1]:.6f}")

            if val_losses[-1] < best:
                best, wait = val_losses[-1], 0
                torch.save(model.state_dict(), EXP / "best_model.pt")
            else:
                wait += 1
                if wait >= PATIENCE:
                    break

        train_time = time.time() - t0

        # ---------- Plot Train/Val Loss ----------
        plt.figure()
        plt.plot(train_losses, label="Train")
        plt.plot(val_losses, label="Val")
        plt.xlabel("Epoch")
        plt.ylabel("MSE Loss")
        plt.legend()
        plt.title(f"{tag} | LB={L} | H={H}")
        plt.savefig(EXP / "train_val_loss.png", dpi=300, bbox_inches="tight")
        plt.close()

        # ---------- Test ----------
        model.load_state_dict(torch.load(EXP / "best_model.pt"))
        model.eval()

        preds, acts = [], []
        t1 = time.time()
        with torch.no_grad():
            for b in test_loader:
                b = b.to(device)
                preds.append(model(b).cpu().numpy())
                acts.append(b.y.view(-1, H).cpu().numpy())
        inf_time = (time.time() - t1) / len(test_loader.dataset)

        preds = np.vstack(preds)
        acts  = np.vstack(acts)

        tmin, tmax = scaler.data_min_[tgt_idx], scaler.data_max_[tgt_idx]
        preds = preds * (tmax - tmin) + tmin
        acts  = acts * (tmax - tmin) + tmin

        y_true, y_pred = acts.flatten(), preds.flatten()
        errors = y_true - y_pred

        # ---------- Step-wise MAPE ----------
        step_mapes = []
        for s in range(H):
            yt = acts[:, s]
            yp = preds[:, s]
            step_mapes.append(np.mean(np.abs((yt - yp) / (yt + 1e-12))) * 100)

        auhe = compute_auhe(step_mapes)

        step_df = pd.DataFrame({
            "Step": np.arange(1, H + 1),
            "MAPE": step_mapes,
            "Lookback": L,
            "Horizon": H,
            "Input_Set": tag
        })
        step_df.to_csv(EXP / "stepwise_mape.csv", index=False)

        plt.figure()
        plt.plot(range(1, H + 1), step_mapes, marker="o")
        plt.xlabel("Forecast Step")
        plt.ylabel("Average MAPE (%)")
        plt.title(f"Step-wise MAPE | LB={L} | H={H} | {tag}")
        plt.grid(True)
        plt.savefig(EXP / "stepwise_mape.png", dpi=300, bbox_inches="tight")
        plt.close()

        # ---------- DM Test ----------
        if tag == "FULL":
            reference_errors[(L, H)] = errors
        elif (L, H) in reference_errors:
            dm, p = diebold_mariano(reference_errors[(L, H)], errors)
            dm_rows.append({
                "Lookback": L,
                "Horizon": H,
                "Reference": "FULL",
                "Compared_Model": tag,
                "DM_stat": dm,
                "p_value": p,
                "Significant_5pct": p < 0.05,
                "Significant_1pct": p < 0.01
            })

        flops = estimate_gcn_flops(
            num_nodes=len(FEATURES),
            seq_len=L,
            hidden=HIDDEN_CHANNELS,
            horizon=H,
            num_layers=NUM_LAYERS
        )


        metrics = {
            "Input_Set": tag,
            "Lookback": L,
            "Horizon": H,
            "MAE": mean_absolute_error(y_true, y_pred),
            "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
            "MAPE": mean_absolute_percentage_error(y_true, y_pred) * 100,
            "sMAPE": smape(y_true, y_pred),
            "NRMSE": nrmse(y_true, y_pred),
            "Directional_Accuracy": directional_accuracy(y_true, y_pred),
            "AUHE": auhe,
            "Train_Time_sec": train_time,
            "Inference_Time_sec": inf_time,
            "Num_Parameters": count_parameters(model),
            "Model_Size_MB": model_size_mb(model),
            "FLOPs": flops
        }

        pd.DataFrame({"Actual": y_true, "Prediction": y_pred}).to_csv(EXP / "predictions_series.csv", index=False)
        pd.DataFrame([metrics]).to_csv(EXP / "metrics.csv", index=False)
        all_results.append(metrics)

# ===================== Save Global =====================
pd.DataFrame(all_results).to_csv(ROOT / "all_results.csv", index=False)
pd.DataFrame(dm_rows).to_csv(ROOT / "diebold_mariano_summary.csv", index=False)

print("\n✔ ALL experiments completed successfully.")


/home/muktinath/WORKSPACE/.PyTorch/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda

=== Lookback 10 | Horizon 1 ===
▶ Input: w/o_Copper_Lead_Nickel_Tin_Zinc
Epoch 005 | Train 0.003372 | Val 0.017688
Epoch 010 | Train 0.002025 | Val 0.016405
Epoch 015 | Train 0.001530 | Val 0.010924
Epoch 020 | Train 0.001429 | Val 0.004599
Epoch 025 | Train 0.001089 | Val 0.002351
Epoch 030 | Train 0.001126 | Val 0.001198
Epoch 035 | Train 0.001060 | Val 0.001822
Epoch 040 | Train 0.000998 | Val 0.002026
Epoch 045 | Train 0.000999 | Val 0.000739
Epoch 050 | Train 0.000968 | Val 0.000812
Epoch 055 | Train 0.000963 | Val 0.000750
Epoch 060 | Train 0.000905 | Val 0.000847
Epoch 065 | Train 0.000958 | Val 0.000782
Epoch 070 | Train 0.000912 | Val 0.000750
▶ Input: w/o_Lead_Nickel_Tin_Zinc
Epoch 005 | Train 0.004373 | Val 0.014795
Epoch 010 | Train 0.002097 | Val 0.003113
Epoch 015 | Train 0.001711 | Val 0.001422
Epoch 020 | Train 0.001441 | Val 0.001310
Epoch 025 | Train 0.001323 | Val 0.000962
Epoch 030 | Train 0.001459 | Val 0.001027
Epoch 035 | Train 0.001317 | Val 0.0013